In [1]:
from transformers import (
    AutoTokenizer, DataCollatorWithPadding,
    AutoModelForSequenceClassification, TrainingArguments, Trainer
)
from datasets import Dataset
import pandas as pd
import evaluate
import numpy as np

In [2]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

In [3]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [4]:
class CustomDataset(Dataset):
    def __init__(self, df):
        self.df = df
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {'question': row.question, 'label': row.label}

In [5]:
dataset = {}
fnames = ['train', 'val', 'test']
for fname in fnames:
    data = {'label': [], 'question': []}
    with open(f'data/{fname}', 'r') as f:
        for line in f:
            label = int(line.split(' ')[0])
            question = line[2:]
            data['label'].append(label)
            data['question'].append(question)
    dataset[fname] = data

In [6]:
train_df = pd.DataFrame(dataset['train'])
val_df = pd.DataFrame(dataset['val'])
test_df = pd.DataFrame(dataset['test'])

In [7]:
train_df.head()

,label,question
0,1,"certainly , many algorithms are built complex ..."
1,1,but what i would like to ask - what are the al...
2,0,is it just based on the length of code ?\n
3,1,is tere a way to solve a math problem with alg...
4,1,could dna be an algorithm ?\n


In [8]:
train_ds = Dataset.from_dict(dataset['train'])
val_ds = Dataset.from_dict(dataset['val'])
test_ds = Dataset.from_dict(dataset['test'])

In [9]:
def preprocess_function(examples):
    return tokenizer(examples["question"], truncation=True)

In [10]:
tokenized_train = train_ds.map(preprocess_function, batched=True)
tokenized_val = val_ds.map(preprocess_function, batched=True)
tokenized_test = test_ds.map(preprocess_function, batched=True)

  0%|          | 0/6 [00:00<?, ?ba/s]

  0%|          | 0/1 [00:00<?, ?ba/s]

  0%|          | 0/1 [00:00<?, ?ba/s]

In [11]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2, id2label=id2label, label2id=label2id
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
accuracy = evaluate.load("accuracy")

In [14]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [15]:
training_args = TrainingArguments(
    output_dir="question_classification",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
trainer.evaluate(tokenized_test)